In [1]:
# Task 9 - Optimized Classification Model with Feature Importance Analysis
# P. Shadik Khan - G40 AI ML

import os
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib

# Use a non-GUI backend to avoid Tkinter errors
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import joblib

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# Load dataset

DATA_PATH = "../data/ecommerce_customer_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (2000, 19)


,CustomerID,Age,Gender,Location,DeviceType,TrafficSource,PagesViewed,TimeOnSite,ProductsViewed,CartItems,PreviousPurchases,AverageOrderValue,DiscountUsed,EmailClicked,AdClicked,ReviewScoreViewed,DaysSinceLastVisit,SessionCount,Purchase
0,100001,22.0,Male,West,Desktop,Social Media,12,5.33,9,0,1,100.66,0,0,0,5.00,9,4,0
1,100002,55.0,Male,Central,Mobile,Search,4,15.31,3,1,4,127.89,0,0,0,4.16,50,8,0
2,100003,49.0,Male,West,Desktop,Email,10,6.50,6,3,1,27.74,0,0,1,3.31,18,4,0
3,100004,39.0,Male,East,Mobile,Search,9,19.73,5,1,1,87.80,0,0,1,4.60,7,5,0
4,100005,38.0,Female,East,Mobile,Search,10,2.55,6,2,0,55.76,0,0,1,3.41,6,5,0


In [3]:
# Basic dataset information

print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nPurchase distribution:")
print(df["Purchase"].value_counts())

print("\nPurchase percentage:")
print(df["Purchase"].value_counts(normalize=True).mul(100).round(2))

Rows and columns: (2000, 19)

Column names:
['CustomerID', 'Age', 'Gender', 'Location', 'DeviceType', 'TrafficSource', 'PagesViewed', 'TimeOnSite', 'ProductsViewed', 'CartItems', 'PreviousPurchases', 'AverageOrderValue', 'DiscountUsed', 'EmailClicked', 'AdClicked', 'ReviewScoreViewed', 'DaysSinceLastVisit', 'SessionCount', 'Purchase']

Data types:
CustomerID              int64
Age                   float64
Gender                 object
Location               object
DeviceType             object
TrafficSource          object
PagesViewed             int64
TimeOnSite            float64
ProductsViewed          int64
CartItems               int64
PreviousPurchases       int64
AverageOrderValue     float64
DiscountUsed            int64
EmailClicked            int64
AdClicked               int64
ReviewScoreViewed     float64
DaysSinceLastVisit      int64
SessionCount            int64
Purchase                int64
dtype: object

Missing values:
CustomerID             0
Age                   40

In [4]:
# Separate features and target

# CustomerID is an identifier, so we remove it
X = df.drop(columns=["Purchase", "CustomerID"])
y = df["Purchase"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (2000, 17)
Target shape: (2000,)

Target distribution:
Purchase
0    1766
1     234
Name: count, dtype: int64


In [5]:
# Identify numerical and categorical features

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['Age', 'PagesViewed', 'TimeOnSite', 'ProductsViewed', 'CartItems', 'PreviousPurchases', 'AverageOrderValue', 'DiscountUsed', 'EmailClicked', 'AdClicked', 'ReviewScoreViewed', 'DaysSinceLastVisit', 'SessionCount']

Categorical columns:
['Gender', 'Location', 'DeviceType', 'TrafficSource']


In [6]:
# Split the dataset into training and testing sets
# Stratification keeps the purchase ratio similar in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 1600
Testing samples: 400

Training target distribution:
Purchase
0    1413
1     187
Name: count, dtype: int64

Testing target distribution:
Purchase
0    353
1     47
Name: count, dtype: int64


In [7]:
# Numerical preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numerical_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [8]:
# Baseline Logistic Regression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


# Baseline Decision Tree

tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)


# Baseline Random Forest

forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Three baseline models created successfully.")

Three baseline models created successfully.


In [9]:
# Train baseline models

print("Training Logistic Regression...")
logistic_pipeline.fit(X_train, y_train)

print("Training Decision Tree...")
tree_pipeline.fit(X_train, y_train)

print("Training Random Forest...")
forest_pipeline.fit(X_train, y_train)

print("\nAll baseline models trained successfully!")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...

All baseline models trained successfully!


In [10]:
# Function to evaluate classification models

def evaluate_model(model, model_name):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    }

    return results


baseline_results = []

baseline_results.append(
    evaluate_model(
        logistic_pipeline,
        "Logistic Regression"
    )
)

baseline_results.append(
    evaluate_model(
        tree_pipeline,
        "Decision Tree"
    )
)

baseline_results.append(
    evaluate_model(
        forest_pipeline,
        "Random Forest"
    )
)

baseline_df = pd.DataFrame(baseline_results)

display(
    baseline_df.round(4)
)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.8775,0.000,0.0000,0.0000,0.5845
1,Decision Tree,0.7925,0.125,0.1277,0.1263,0.5043
2,Random Forest,0.8825,0.000,0.0000,0.0000,0.5989


In [11]:
# Confusion matrices for baseline models

models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": tree_pipeline,
    "Random Forest": forest_pipeline
}

for name, model in models.items():
    predictions = model.predict(X_test)
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Purchase", "Purchase"],
        yticklabels=["No Purchase", "Purchase"]
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()

    filename = name.lower().replace(" ", "_") + "_confusion_matrix.png"

    plt.savefig(
        f"../reports/{filename}",
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()

print("Confusion matrices saved successfully.")

Confusion matrices saved successfully.


In [12]:
# ROC curves for baseline models

plt.figure(figsize=(8, 6))

for name, model in models.items():

    probabilities = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    auc_score = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline Models")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "../reports/baseline_roc_curve.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print("ROC curve saved successfully.")

ROC curve saved successfully.


In [13]:
# Analyze class imbalance

class_counts = y.value_counts()
class_percentages = y.value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(2)
})

display(imbalance_df)

print(
    f"\nPurchase rate: "
    f"{class_percentages[1]:.2f}%"
)

print(
    f"Non-purchase rate: "
    f"{class_percentages[0]:.2f}%"
)

,Count,Percentage
Purchase,,
0,1766,88.3
1,234,11.7



Purchase rate: 11.70%
Non-purchase rate: 88.30%


In [14]:
# Hyperparameter optimization for Random Forest

forest_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__class_weight": [None, "balanced"]
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

forest_grid = GridSearchCV(
    estimator=forest_pipeline,
    param_grid=forest_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

print("Starting Random Forest GridSearchCV...")

forest_grid.fit(
    X_train,
    y_train
)

print("\nRandom Forest optimization completed!")

print("\nBest parameters:")
print(forest_grid.best_params_)

print(
    "\nBest cross-validation F1-score:",
    round(forest_grid.best_score_, 4)
)

Starting Random Forest GridSearchCV...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Random Forest optimization completed!

Best parameters:
{'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}

Best cross-validation F1-score: 0.0105


In [15]:
# Evaluate optimized Random Forest

best_forest = forest_grid.best_estimator_

optimized_predictions = best_forest.predict(X_test)

optimized_probabilities = (
    best_forest.predict_proba(X_test)[:, 1]
)

optimized_accuracy = accuracy_score(
    y_test,
    optimized_predictions
)

optimized_precision = precision_score(
    y_test,
    optimized_predictions,
    zero_division=0
)

optimized_recall = recall_score(
    y_test,
    optimized_predictions,
    zero_division=0
)

optimized_f1 = f1_score(
    y_test,
    optimized_predictions,
    zero_division=0
)

optimized_auc = roc_auc_score(
    y_test,
    optimized_probabilities
)

print("Optimized Random Forest Results")
print("--------------------------------")
print("Accuracy :", round(optimized_accuracy, 4))
print("Precision:", round(optimized_precision, 4))
print("Recall   :", round(optimized_recall, 4))
print("F1-Score :", round(optimized_f1, 4))
print("ROC-AUC  :", round(optimized_auc, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        optimized_predictions,
        zero_division=0
    )
)

Optimized Random Forest Results
--------------------------------
Accuracy : 0.8775
Precision: 0.0
Recall   : 0.0
F1-Score : 0.0
ROC-AUC  : 0.5667

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.99      0.93       353
           1       0.00      0.00      0.00        47

    accuracy                           0.88       400
   macro avg       0.44      0.50      0.47       400
weighted avg       0.78      0.88      0.82       400



In [16]:
# Baseline vs optimized model comparison

optimized_result = pd.DataFrame([{
    "Model": "Optimized Random Forest",
    "Optimization Status": "Optimized",
    "Accuracy": optimized_accuracy,
    "Precision": optimized_precision,
    "Recall": optimized_recall,
    "F1-Score": optimized_f1,
    "ROC-AUC": optimized_auc
}])

baseline_comparison = baseline_df.copy()

baseline_comparison.insert(
    1,
    "Optimization Status",
    "Baseline"
)

final_model_comparison = pd.concat(
    [
        baseline_comparison,
        optimized_result
    ],
    ignore_index=True
)

display(
    final_model_comparison.round(4)
)

final_model_comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Model comparison saved.")

,Model,Optimization Status,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Baseline,0.8775,0.000,0.0000,0.0000,0.5845
1,Decision Tree,Baseline,0.7925,0.125,0.1277,0.1263,0.5043
2,Random Forest,Baseline,0.8825,0.000,0.0000,0.0000,0.5989
3,Optimized Random Forest,Optimized,0.8775,0.000,0.0000,0.0000,0.5667


Model comparison saved.
